In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)

df = pd.read_csv("hotel_bookings_clean.csv")

print("Shape:", df.shape)

df.head()

Shape: (87377, 37)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,total_guests,total_nights,family,arrival_month_num,season
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,0.0,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01,2.0,0,0,7,Summer
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,0.0,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01,2.0,0,0,7,Summer
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,0.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02,1.0,1,0,7,Summer
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02,1.0,1,0,7,Summer
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03,2.0,2,0,7,Summer


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 87377 entries, 0 to 87376
Data columns (total 37 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   hotel                           87377 non-null  str    
 1   is_canceled                     87377 non-null  int64  
 2   lead_time                       87377 non-null  int64  
 3   arrival_date_year               87377 non-null  int64  
 4   arrival_date_month              87377 non-null  str    
 5   arrival_date_week_number        87377 non-null  int64  
 6   arrival_date_day_of_month       87377 non-null  int64  
 7   stays_in_weekend_nights         87377 non-null  int64  
 8   stays_in_week_nights            87377 non-null  int64  
 9   adults                          87377 non-null  int64  
 10  children                        87377 non-null  float64
 11  babies                          87377 non-null  int64  
 12  meal                            87377 non-n

In [3]:
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

Missing values: 82122


Duplicate rows: 0


In [4]:
df['total_guests'] = (
    df['adults'] +
    df['children'] +
    df['babies']
)

In [5]:
df['total_nights'] = (
    df['stays_in_weekend_nights'] +
    df['stays_in_week_nights']
)

In [6]:
df['is_family'] = np.where(
    (df['children'] > 0) | (df['babies'] > 0),
    1,
    0
)

In [7]:
df['has_weekend_stay'] = np.where(
    df['stays_in_weekend_nights'] > 0,
    1,
    0
)

In [8]:
df['estimated_revenue'] = (
    df['adr'] *
    df['total_nights']
)

In [9]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'


df['season'] = df['arrival_month_num'].apply(get_season)

In [10]:
df['season'].value_counts()

season
Summer    29078
Spring    23770
Autumn    18617
Winter    15912
Name: count, dtype: int64

In [11]:
def booking_size(guests):

    if guests == 1:
        return 'Single'

    elif guests == 2:
        return 'Couple'

    elif guests <= 4:
        return 'Small Group'

    else:
        return 'Large Group'


df['booking_size'] = df['total_guests'].apply(booking_size)

In [12]:
df['booking_size'].value_counts()

booking_size
Couple         57050
Single         16060
Small Group    14114
Large Group      153
Name: count, dtype: int64

In [13]:
def stay_category(nights):

    if nights <= 2:
        return 'Short Stay'

    elif nights <= 5:
        return 'Medium Stay'

    else:
        return 'Long Stay'


df['stay_category'] = df['total_nights'].apply(stay_category)

In [14]:
def lead_time_category(days):

    if days <= 7:
        return 'Last Minute'

    elif days <= 30:
        return 'Short Lead Time'

    elif days <= 90:
        return 'Medium Lead Time'

    else:
        return 'Long Lead Time'


df['lead_time_category'] = df['lead_time'].apply(
    lead_time_category
)

In [15]:
df['lead_time_category'].value_counts()

lead_time_category
Long Lead Time      30008
Medium Lead Time    22741
Last Minute         18293
Short Lead Time     16335
Name: count, dtype: int64

In [16]:
leakage_columns = [
    'reservation_status',
    'reservation_status_date'
]

df_ml = df.drop(
    columns=leakage_columns,
    errors='ignore'
)

In [17]:
df_ml = df_ml.drop(
    columns=['agent'],
    errors='ignore'
)

In [18]:
df.drop(columns=['company'])

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,total_guests,total_nights,family,arrival_month_num,season,is_family,has_weekend_stay,estimated_revenue,booking_size,stay_category,lead_time_category
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,0.0,0,Transient,0.00,0,0,Check-Out,2015-07-01,2.0,0,0,7,Summer,0,0,0.00,Couple,Short Stay,Long Lead Time
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,0.0,0,Transient,0.00,0,0,Check-Out,2015-07-01,2.0,0,0,7,Summer,0,0,0.00,Couple,Short Stay,Long Lead Time
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,0.0,0,Transient,75.00,0,0,Check-Out,2015-07-02,1.0,1,0,7,Summer,0,0,75.00,Single,Short Stay,Last Minute
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,0,Transient,75.00,0,0,Check-Out,2015-07-02,1.0,1,0,7,Summer,0,0,75.00,Single,Short Stay,Short Lead Time
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,0,Transient,98.00,0,1,Check-Out,2015-07-03,2.0,2,0,7,Summer,0,0,196.00,Couple,Short Stay,Short Lead Time
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87372,City Hotel,0,23,2017,August,35,30,2,5,2,0.0,0,BB,BEL,Offline TA/TO,TA/TO,0,0,0,A,A,0,No Deposit,394.0,0,Transient,96.14,0,0,Check-Out,2017-09-06,2.0,7,0,8,Summer,0,1,672.98,Couple,Long Stay,Short Lead Time
87373,City Hotel,0,102,2017,August,35,31,2,5,3,0.0,0,BB,FRA,Online TA,TA/TO,0,0,0,E,E,0,No Deposit,9.0,0,Transient,225.43,0,2,Check-Out,2017-09-07,3.0,7,0,8,Summer,0,1,1578.01,Small Group,Long Stay,Long Lead Time
87374,City Hotel,0,34,2017,August,35,31,2,5,2,0.0,0,BB,DEU,Online TA,TA/TO,0,0,0,D,D,0,No Deposit,9.0,0,Transient,157.71,0,4,Check-Out,2017-09-07,2.0,7,0,8,Summer,0,1,1103.97,Couple,Long Stay,Medium Lead Time
87375,City Hotel,0,109,2017,August,35,31,2,5,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,89.0,0,Transient,104.40,0,0,Check-Out,2017-09-07,2.0,7,0,8,Summer,0,1,730.80,Couple,Long Stay,Long Lead Time


In [19]:
X = df_ml.drop(
    columns=['is_canceled']
)

y = df_ml['is_canceled']

In [20]:
print("Features:", X.shape)
print("Target:", y.shape)

Features: (87377, 39)
Target: (87377,)


In [21]:
categorical_features = X.select_dtypes(
    include=['object']
).columns.tolist()

categorical_features

C:\Users\Ankit\AppData\Local\Temp\ipykernel_11312\1607483279.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


['hotel',
 'arrival_date_month',
 'meal',
 'country',
 'market_segment',
 'distribution_channel',
 'reserved_room_type',
 'assigned_room_type',
 'deposit_type',
 'customer_type',
 'season',
 'booking_size',
 'stay_category',
 'lead_time_category']

In [22]:
numerical_features = X.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

numerical_features

['lead_time',
 'arrival_date_year',
 'arrival_date_week_number',
 'arrival_date_day_of_month',
 'stays_in_weekend_nights',
 'stays_in_week_nights',
 'adults',
 'children',
 'babies',
 'is_repeated_guest',
 'previous_cancellations',
 'previous_bookings_not_canceled',
 'booking_changes',
 'company',
 'days_in_waiting_list',
 'adr',
 'required_car_parking_spaces',
 'total_of_special_requests',
 'total_guests',
 'total_nights',
 'family',
 'arrival_month_num',
 'is_family',
 'has_weekend_stay',
 'estimated_revenue']

In [23]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [24]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'categorical',
            OneHotEncoder(
                handle_unknown='ignore'
            ),
            categorical_features
        )
    ],
    remainder='passthrough'
)

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [26]:
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("\nTraining target:")
print(y_train.value_counts(normalize=True))

print("\nTesting target:")
print(y_test.value_counts(normalize=True))

Training data: (69901, 39)
Testing data: (17476, 39)

Training target:
is_canceled
0    0.725054
1    0.274946
Name: proportion, dtype: float64

Testing target:
is_canceled
0    0.725051
1    0.274949
Name: proportion, dtype: float64


In [27]:
df.to_csv(
    "hotel_bookings_engineered.csv",
    index=False
)

In [28]:
df_ml.to_csv(
    "hotel_bookings_ml.csv",
    index=False
)

In [29]:
print("=" * 50)
print("FINAL DATASET CHECK")
print("=" * 50)

print("Rows:", df_ml.shape[0])
print("Columns:", df_ml.shape[1])
print("Missing values:", df_ml.isnull().sum().sum())
print("Duplicates:", df_ml.duplicated().sum())
print("Target distribution:")
print(y.value_counts())

FINAL DATASET CHECK
Rows: 87377
Columns: 40


Missing values: 82122


Duplicates: 276
Target distribution:
is_canceled
0    63353
1    24024
Name: count, dtype: int64
